# Classificação Fine-Grained de Raças de Cães

**Trabalho final — Visão Computacional**
Pós-graduação em LLM e IA Generativa

Autores: Victor Macaubas e Mari
Data: agosto/2026

Repositório: https://github.com/victormacaubas/project-image-processing

---

> **Este notebook é autossuficiente e roda em ~2 minutos.**
>
> Basta `Ambiente de execução → Executar tudo`. A primeira célula clona o
> repositório e instala o que falta; nenhum arquivo adicional é necessário e
> nada precisa ser baixado.
>
> Os resultados, gráficos e a análise de erros são reconstruídos a partir das
> predições (logits) versionadas no repositório — por isso é rápido.
>
> Para reexecutar os treinos do zero, troque `RETRAIN` para `True` na célula de
> setup. Aí sim leva ~2 horas com GPU.

In [ ]:
# ─── Bootstrap ──────────────────────────────────────────────────────────────
# Deixa o notebook autossuficiente: clona o repositório e instala o que falta.
# Idempotente — rodar duas vezes não causa dano.
#
# NÃO reinstala torch/torchvision: o Colab já os traz, e reinstalar dispara
# "restart runtime", que aborta a execução de ponta a ponta.

REPO_URL = "https://github.com/victormacaubas/project-image-processing.git"
REPO_NAME = "project-image-processing"

import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def _run(cmd: list[str]) -> None:
    """Roda o comando e, se falhar, mostra a saída real.

    Sem isso, um pip que falha com -q some silenciosamente e o erro só
    aparece 20 células depois, como ImportError sem contexto.
    """
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr, file=sys.stderr)
        raise RuntimeError(f"Falhou: {' '.join(cmd)}")


def _find_repo_root() -> Path:
    """Localiza a raiz do repo, clonando se necessário. Idempotente."""
    here = Path.cwd()

    # Já estamos dentro do repositório?
    for candidate in (here, *here.parents):
        if (candidate / "src" / "dogs" / "config.py").exists():
            return candidate

    # Já clonado num subdiretório?
    if (here / REPO_NAME / "src" / "dogs" / "config.py").exists():
        return here / REPO_NAME

    # Clonar.
    print(f"Clonando {REPO_URL} ...")
    _run(["git", "clone", "--depth", "1", REPO_URL, REPO_NAME])
    return here / REPO_NAME


REPO_ROOT = _find_repo_root()

if IN_COLAB:
    reqs = REPO_ROOT / "requirements-colab.txt"
    if reqs.exists():
        print("Instalando dependências ...")
        _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(reqs)])

# Torna o pacote `dogs` importável, sem duplicar entradas em sys.path.
src = str(REPO_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)

print(f"Repositório: {REPO_ROOT}")

In [ ]:
# ─── Verificação ────────────────────────────────────────────────────────────
# Falha aqui, alto e claro, em vez de estourar um erro críptico 20 células
# adiante. Se esta célula passar, o notebook roda até o fim.

import importlib

problemas = []

for pacote in ["torch", "torchvision", "numpy", "pandas", "sklearn",
               "matplotlib", "seaborn", "datasets"]:
    try:
        importlib.import_module(pacote)
    except ImportError as erro:
        problemas.append(f"pacote ausente: {pacote} ({erro})")

try:
    from dogs.config import describe_environment, ensure_dirs
    ensure_dirs()
    print(describe_environment())
except Exception as erro:
    problemas.append(f"pacote `dogs` não importável: {erro}")

if problemas:
    raise RuntimeError(
        "Ambiente incompleto:\n  - " + "\n  - ".join(problemas)
        + "\n\nRode a célula de bootstrap acima antes desta."
    )

print("\nAmbiente OK.")

In [ ]:
# ─── Setup ──────────────────────────────────────────────────────────────────
# False -> reconstrói tudo das predições versionadas (~2 min). É o padrão.
# True  -> reexecuta os treinos a partir do dataset (~2 h com GPU).
RETRAIN = False

import logging
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns

from dogs.config import (TrainConfig, FEATURES_DIR, CHECKPOINT_DIR,
                         PREDICTIONS_DIR, RESULTS_CSV)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
torch.manual_seed(42)
np.random.seed(42)

### Artefatos

Este notebook roda em três níveis, do mais leve ao mais completo.

| Nível | O que precisa | De onde vem | Tempo |
|---|---|---|---|
| **Padrão** (`RETRAIN = False`) | predições (~3 MB cada) | já vêm no repositório | ~2 min |
| Reproduzir E2 | embeddings (~170 MB) | Drive público | ~10 min |
| Reproduzir tudo | nada | dataset original | ~2 h com GPU |

No nível padrão **nada é baixado**: as predições (logits + labels de cada
experimento) estão versionadas no Git, e delas saem a tabela de resultados, os
gráficos e a análise de erros inteira. Um checkpoint de ResNet50 tem ~100 MB; os
logits que ele produz, ~3 MB. Para reproduzir a *análise*, os logits bastam.

Os pesos treinados continuam disponíveis no Drive para quem quiser reexecutar os
modelos — a célula abaixo os busca quando `RETRAIN = True`.

In [ ]:
# ─── Artefatos pesados (só quando RETRAIN = True) ───────────────────────────
# Pasta do Drive com embeddings e checkpoints.
# LEMBRETE(sexta): antes de entregar, tornar a pasta pública —
#   Compartilhar -> Acesso geral -> "Qualquer pessoa com o link" -> Leitor.
#   Enquanto estiver restrita, só funciona para quem tem acesso concedido.
DRIVE_FOLDER_ID = "12XC2LBi7NK02kTdIMN6FAU_Wf7sbKYZ8"

from pathlib import Path

# Assinaturas de arquivo. O Google intercepta downloads grandes com uma página
# de aviso de vírus; sem esta checagem, o gdown grava esse HTML com nome .npy e
# o erro só aparece muito depois, como um "corrupt file" incompreensível.
ASSINATURAS = {".npy": b"\x93NUMPY", ".npz": b"PK", ".pt": b"PK"}


def arquivo_valido(caminho: Path) -> bool:
    esperado = ASSINATURAS.get(caminho.suffix)
    if esperado is None:
        return True
    with caminho.open("rb") as f:
        return f.read(len(esperado)) == esperado


def validar_diretorio(diretorio: Path) -> list[Path]:
    """Devolve os arquivos corrompidos (normalmente HTML disfarçado)."""
    return [
        p for p in diretorio.rglob("*")
        if p.is_file() and p.suffix in ASSINATURAS and not arquivo_valido(p)
    ]


def baixar_artefatos() -> None:
    import gdown
    destino = FEATURES_DIR.parent
    gdown.download_folder(
        id=DRIVE_FOLDER_ID, output=str(destino), quiet=False, use_cookies=False
    )

    if corrompidos := validar_diretorio(destino):
        for p in corrompidos:
            p.unlink()
        raise RuntimeError(
            f"{len(corrompidos)} arquivo(s) baixados como HTML, não como dados.\n"
            "Causa provável: a pasta do Drive não está pública, ou o Google "
            "interceptou o download com a tela de aviso de vírus (comum acima "
            "de 100 MB).\n"
            "Saídas: (1) conferir que o acesso é 'Qualquer pessoa com o link'; "
            "(2) publicar os artefatos numa GitHub Release, que não tem esse "
            "limite; (3) rodar com RETRAIN = False."
        )


if RETRAIN and not (FEATURES_DIR / "resnet50_train_X.npy").exists():
    baixar_artefatos()

# No modo padrão, confere que as predições vieram junto com o repositório.
if not RETRAIN:
    encontradas = sorted(PREDICTIONS_DIR.glob("*.npz"))
    if not encontradas:
        raise RuntimeError(
            "Nenhuma predição em reports/predictions/.\n"
            "Elas deveriam vir versionadas no repositório. Rode com "
            "RETRAIN = True para regerá-las a partir do dataset."
        )
    print(f"{len(encontradas)} predições disponíveis:")
    for p in encontradas:
        print(f"  {p.stem}  ({p.stat().st_size / 1e6:.1f} MB)")

---
# 1. Descrição do problema

<!-- OBRIGATÓRIO NA ENTREGA — dona: Mari, prazo: quinta -->

**Escrever aqui:**

- O que é classificação fine-grained e por que difere de classificação genérica
- Por que é difícil: baixa variância entre classes, alta variância dentro da classe
- Por que importa (aplicações reais)
- A pergunta que guia o trabalho: *quanto de representação visual precisa ser
  aprendido versus transferido, com dados limitados?*
- O que consideramos sucesso

---
# 2. Descrição da base de dados

<!-- OBRIGATÓRIO NA ENTREGA — dona: Mari, prazo: quinta -->

**Escrever aqui:** origem, licença, 20.580 imagens, 120 classes, split oficial
12.000/8.580, como separamos validação, distribuição de classes, resolução das imagens.

**Não esquecer:** Stanford Dogs é derivado do ImageNet. Backbones pré-treinados em
ImageNet já viram essas imagens. Discutir na seção 6.

In [ ]:
# EDA — colar do notebooks/01_eda.ipynb
# Distribuição de classes, grid de amostras, exemplos de raças visualmente próximas

---
# 3. Metodologia

<!-- OBRIGATÓRIO NA ENTREGA — donos: Mari e Victor, prazo: quinta -->

**Escrever aqui:**

- Estratégia geral: progressão do zero → transferido → adaptado
- Pré-processamento e augmentation (e por que só no treino)
- Arquiteturas de cada experimento
- Protocolo de treino: AdamW, cosine schedule, early stopping na val, label smoothing
- Métricas: top-1, top-5, F1 macro — e por que cada uma
- **Por que pré-computamos embeddings:** decisão de engenharia que viabilizou o prazo

## 3.1 Extração de embeddings

Passo executado uma vez, fora do notebook:

```bash
python -m dogs.features --backbone resnet50
```

Gera `{backbone}_{split}_{X,y}.npy` em `data/processed/features/`.

In [ ]:
def load_features(backbone="resnet50", split="train"):
    X = np.load(FEATURES_DIR / f"{backbone}_{split}_X.npy")
    y = np.load(FEATURES_DIR / f"{backbone}_{split}_y.npy")
    return X, y

X_train, y_train = load_features(split="train")
X_val, y_val = load_features(split="val")
print(X_train.shape, X_val.shape)

---
# 4. Experimentos

<!-- OBRIGATÓRIO NA ENTREGA -->

Cada experimento: o que testa, como foi configurado, o que aconteceu.

## E1 — CNN treinada do zero

**Hipótese:** sem transferência, 120 classes fine-grained com ~100 imagens de treino
por classe não dão sinal suficiente. Esperamos acurácia baixa.

*Dona: Mari*

In [ ]:
from dogs.models import SmallCNN
# ...

## E2 — Linear probe sobre backbone congelado

**Hipótese:** a representação do ImageNet já separa bem as raças, mesmo sem nenhuma
adaptação — um classificador linear deve superar E1 por larga margem.

*Dona: Mari*

In [ ]:
from dogs.models import LinearProbe
# ...

## E3 — Fine-tuning parcial

**Hipótese:** descongelar os blocos finais permite adaptar as features de alto nível
ao domínio e deve superar E2.

*Dono: Victor*

In [ ]:
from dogs.models import build_finetune_model
# ...

---
# 5. Resultados

<!-- OBRIGATÓRIO NA ENTREGA -->

In [ ]:
results = pd.read_csv(RESULTS_CSV)
results.sort_values("top1", ascending=False)

In [ ]:
# Gráfico comparativo top-1 por experimento

---
# 6. Análise

<!-- OBRIGATÓRIO NA ENTREGA — dono: Victor, prazo: quinta -->

## 6.1 O salto do transfer learning

Comparar E1 vs. E2 vs. E3 e interpretar a magnitude da diferença.

## 6.2 Quais raças o modelo confunde

Pares mais confundidos. As confusões são visualmente plausíveis? Um humano erraria
os mesmos casos?

## 6.3 ⚠️ Contaminação entre Stanford Dogs e ImageNet

Stanford Dogs foi construído a partir do ImageNet. O backbone pré-treinado em
ImageNet-1k **já viu essas imagens**. Nossos números de transfer learning são,
portanto, otimistas e não estimam o desempenho em um domínio novo.

Discutir: o que isso invalida, o que continua válido, e como um experimento futuro
poderia medir o efeito (ex.: avaliar em fotos de cães fora do ImageNet).

## 6.4 Limitações

Orçamento computacional, ausência de busca de hiperparâmetros, execução única
por experimento (sem barras de erro), split de teste tocado uma só vez.

In [ ]:
from dogs.evaluate import most_confused_pairs
# ...

---
# 7. Conclusões

<!-- OBRIGATÓRIO NA ENTREGA — dona: Mari, prazo: sexta -->

**Escrever aqui:**

- Resposta direta à pergunta da seção 1
- O que os números mostraram, incluindo o que surpreendeu
- O que faríamos com mais tempo (E5, E6, métodos fine-grained com atenção por partes)

---
## Referências

- Khosla et al. (2011). *Novel Dataset for Fine-Grained Image Categorization: Stanford Dogs.*
- He et al. (2016). *Deep Residual Learning for Image Recognition.*
- Radford et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision.*